# FFT와 Butterfly 연산 이해하기

이 노트북에서는 Additive FFT를 이해하기 전에 알아야 할 **일반 FFT(Cooley-Tukey FFT)**의 원리를 다룹니다.

## 목차
1. 다항식 곱셈이 왜 중요한가
2. 나이브 다항식 곱셈 vs FFT 기반 곱셈
3. Root of Unity
4. DFT (Discrete Fourier Transform)
5. FFT와 Butterfly 연산
6. FFT를 이용한 다항식 곱셈
7. 왜 $\mathbb{F}_2$에서는 안 되는가

## 1. 다항식 곱셈이 왜 중요한가

### 암호학에서의 다항식 곱셈

Post-Quantum Cryptography(PQC)의 핵심 연산들은 대부분 **다항식 곱셈**으로 귀결됩니다:

| 암호 스킴 | 링 | 다항식 곱셈 위치 |
|-----------|-----|------------------|
| ML-KEM (Kyber) | $\mathbb{Z}_q[x]/(x^n+1)$ | Key generation, Encap, Decap |
| HQC | $\mathbb{F}_2[x]/(x^n-1)$ | Key generation, Encap, Decap |
| BIKE | $\mathbb{F}_2[x]/(x^n-1)$ | Key generation, Encap, Decap |

HQC의 경우 $n = 17669$ (보안 레벨 1)이므로, 17669-bit짜리 다항식 두 개를 곱해야 합니다.
이를 효율적으로 하는 것이 전체 성능의 핵심입니다.

### 다항식 곱셈의 기본 아이디어

두 다항식 $A(x)$와 $B(x)$의 곱 $C(x) = A(x) \cdot B(x)$를 구하는 방법은?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 예시 다항식
# A(x) = 2 + 3x + x^2
# B(x) = 1 + 2x
A = [2, 3, 1]    # 계수: a_0=2, a_1=3, a_2=1
B = [1, 2]        # 계수: b_0=1, b_1=2

def poly_to_str(coeffs):
    """계수 리스트를 다항식 문자열로 변환"""
    terms = []
    for i, c in enumerate(coeffs):
        if c == 0:
            continue
        if i == 0:
            terms.append(f"{c}")
        elif i == 1:
            terms.append(f"{c}x" if c != 1 else "x")
        else:
            terms.append(f"{c}x^{i}" if c != 1 else f"x^{i}")
    return " + ".join(terms) if terms else "0"

print(f"A(x) = {poly_to_str(A)}")
print(f"B(x) = {poly_to_str(B)}")
print(f"\n기대하는 결과: C(x) = A(x)·B(x) = (2+3x+x²)(1+2x)")
print(f"  = 2·1 + (2·2+3·1)x + (3·2+1·1)x² + (1·2)x³")
print(f"  = 2 + 7x + 7x² + 2x³")

## 2. 나이브 다항식 곱셈 vs FFT 기반 곱셈

### 방법 1: 나이브 곱셈 (Schoolbook)

모든 계수 쌍 $(a_i, b_j)$를 곱해서 $c_{i+j}$에 더합니다.

- $A(x)$의 차수: $n-1$, $B(x)$의 차수: $m-1$
- 필요한 곱셈 수: $n \times m$ → **$O(n^2)$**

In [ ]:
def naive_multiply(A, B):
    """나이브(schoolbook) 다항식 곱셈. O(n^2)"""
    n, m = len(A), len(B)
    C = [0] * (n + m - 1)
    
    operations = 0
    print("나이브 곱셈 과정:")
    for i in range(n):
        for j in range(m):
            product = A[i] * B[j]
            C[i + j] += product
            operations += 1
            print(f"  a_{i}·b_{j} = {A[i]}·{B[j]} = {product}  → c_{i+j}에 더함")
    
    print(f"\n총 곱셈 횟수: {operations}")
    return C

C_naive = naive_multiply(A, B)
print(f"\n결과: C(x) = {poly_to_str(C_naive)}")

### 방법 2: FFT 기반 곱셈 — 핵심 아이디어

다항식의 중요한 성질을 이용합니다:

> **차수 $n-1$인 다항식은 $n$개의 서로 다른 점에서의 값으로 유일하게 결정된다.**

따라서 다항식 곱셈을 다음과 같이 할 수 있습니다:

```
계수 표현          →    값 표현         →   계수 표현
[a_0, a_1, ...]   →  [A(p_0), A(p_1), ...]  
[b_0, b_1, ...]   →  [B(p_0), B(p_1), ...]  
                         ↓ pointwise multiply
                      [C(p_0), C(p_1), ...]  →  [c_0, c_1, ...]
```

- **Evaluate**: 다항식을 $N$개 점에서 평가
- **Pointwise Multiply**: 점별 곱셈 ($N$번) → $O(N)$
- **Interpolate**: 값으로부터 다항식 복원

핵심: Evaluate와 Interpolate를 **$O(N \log N)$**에 할 수 있으면 전체가 $O(N \log N)$!

In [ ]:
def eval_poly(coeffs, x):
    """다항식을 점 x에서 평가 (Horner's method)"""
    result = 0
    for c in reversed(coeffs):
        result = result * x + c
    return result

# 시연: evaluate → pointwise multiply → interpolate
print("FFT 기반 곱셈의 개념 시연")
print("=" * 50)

# C(x)의 차수는 최대 deg(A) + deg(B) = 2 + 1 = 3, 계수 4개
# 따라서 최소 4개의 점이 필요
points = [0, 1, 2, 3]

print("\nStep 1: Evaluate — 각 점에서 A(x), B(x) 평가")
A_vals = [eval_poly(A, p) for p in points]
B_vals = [eval_poly(B, p) for p in points]
for i, p in enumerate(points):
    print(f"  x={p}: A({p})={A_vals[i]}, B({p})={B_vals[i]}")

print("\nStep 2: Pointwise Multiply — 점별 곱셈")
C_vals = [a * b for a, b in zip(A_vals, B_vals)]
for i, p in enumerate(points):
    print(f"  x={p}: C({p}) = A({p})·B({p}) = {A_vals[i]}·{B_vals[i]} = {C_vals[i]}")

print("\nStep 3: Interpolate — 값으로부터 다항식 복원")
# numpy로 Lagrange interpolation
C_interp = np.round(np.polynomial.polynomial.polyfit(points, C_vals, 3)).astype(int)
print(f"  복원된 C(x) = {poly_to_str(list(C_interp))}")
print(f"  나이브 결과  = {poly_to_str(C_naive)}")
print(f"  일치: {'✓' if list(C_interp) == C_naive else '✗'}")

### 문제: 아무 점이나 고르면 Evaluate/Interpolate가 $O(n^2)$

임의의 점에서 evaluate하면 Vandermonde 행렬을 풀어야 해서 $O(n^2)$입니다.

**특별한 점**을 골라야 $O(n \log n)$이 가능합니다 → 이것이 **root of unity**입니다!

## 3. Root of Unity (단위근)

$n$-th root of unity $\omega$는 다음을 만족하는 수입니다:

$$\omega^n = 1, \quad \omega^k \neq 1 \text{ for } 0 < k < n$$

복소수에서: $\omega = e^{2\pi i / n}$

### Root of Unity의 핵심 성질

1. **$n$개의 서로 다른 점**: $\{1, \omega, \omega^2, \ldots, \omega^{n-1}\}$
2. **대칭성**: $\omega^{n/2} = -1$ (cancellation lemma)
3. **제곱 축소**: $\{\omega^0, \omega^1, \ldots, \omega^{n-1}\}$의 제곱 = $\{(\omega^2)^0, (\omega^2)^1, \ldots, (\omega^2)^{n/2-1}\}$ (절반으로 줄어듦)

성질 2, 3이 **butterfly 연산**과 **분할 정복**을 가능하게 합니다.

In [ ]:
# Root of Unity 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, n in enumerate([4, 8, 16]):
    ax = axes[idx]
    
    # n-th roots of unity
    angles = [2 * np.pi * k / n for k in range(n)]
    roots = [np.exp(1j * angle) for angle in angles]
    
    # 단위원
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'lightgray', linewidth=1)
    ax.axhline(y=0, color='lightgray', linewidth=0.5)
    ax.axvline(x=0, color='lightgray', linewidth=0.5)
    
    # roots 그리기
    for k, root in enumerate(roots):
        color = 'red' if k == 0 else ('blue' if k == n//2 else 'black')
        ax.plot(root.real, root.imag, 'o', color=color, markersize=8)
        offset = 0.15
        ax.annotate(f'$\\omega^{{{k}}}$', (root.real + offset * np.cos(angles[k]), 
                    root.imag + offset * np.sin(angles[k])), fontsize=8, ha='center')
    
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.set_aspect('equal')
    ax.set_title(f'{n}-th roots of unity\n$\\omega = e^{{2\\pi i/{n}}}$, $\\omega^{{{n//2}}} = -1$')

plt.tight_layout()
plt.savefig('/Users/jaeho/project/FAFFT/notebooks/roots_of_unity.png', dpi=100, bbox_inches='tight')
plt.show()
print("핵심: ω^(n/2) = -1 → butterfly에서 덧셈/뺄셈으로 분할 가능")

## 4. DFT (Discrete Fourier Transform)

다항식 $f(x) = f_0 + f_1 x + f_2 x^2 + \cdots + f_{n-1} x^{n-1}$을
$n$-th roots of unity $\{1, \omega, \omega^2, \ldots, \omega^{n-1}\}$에서 evaluate하는 것이 **DFT**입니다:

$$\hat{f}_k = f(\omega^k) = \sum_{j=0}^{n-1} f_j \cdot \omega^{jk}$$

행렬로 쓰면:

$$\begin{pmatrix} \hat{f}_0 \\ \hat{f}_1 \\ \hat{f}_2 \\ \hat{f}_3 \end{pmatrix} = 
\begin{pmatrix} 1 & 1 & 1 & 1 \\ 1 & \omega & \omega^2 & \omega^3 \\ 1 & \omega^2 & \omega^4 & \omega^6 \\ 1 & \omega^3 & \omega^6 & \omega^9 \end{pmatrix}
\begin{pmatrix} f_0 \\ f_1 \\ f_2 \\ f_3 \end{pmatrix}$$

이 행렬 곱을 나이브하게 하면 $O(n^2)$입니다. **FFT**는 이를 $O(n \log n)$으로 줄입니다.

In [ ]:
def dft_naive(f):
    """나이브 DFT 구현 — O(n^2)"""
    n = len(f)
    omega = np.exp(2j * np.pi / n)  # primitive n-th root of unity
    
    result = np.zeros(n, dtype=complex)
    for k in range(n):
        for j in range(n):
            result[k] += f[j] * omega ** (j * k)
    return result

# 예시: f(x) = 1 + 2x + 3x^2 + 4x^3
f = np.array([1, 2, 3, 4], dtype=complex)
n = len(f)
omega = np.exp(2j * np.pi / n)

print(f"다항식: f(x) = {poly_to_str([1,2,3,4])}")
print(f"n = {n}, ω = e^(2πi/{n})")
print()

# DFT 행렬 시각화
print("DFT 행렬 (n=4):")
print("┌                                    ┐")
for k in range(n):
    row = []
    for j in range(n):
        exp = (k * j) % n
        row.append(f"ω^{exp:1d}")
    print(f"│ {' '.join(f'{r:>4}' for r in row)} │")
print("└                                    ┘")

print(f"\nω^0 = 1, ω^1 = i, ω^2 = -1, ω^3 = -i  (n=4일 때)")

# DFT 계산
F = dft_naive(f)
print("\nDFT 결과:")
for k in range(n):
    print(f"  f(ω^{k}) = {F[k]:.4f}")

# numpy FFT와 비교
F_numpy = np.fft.fft(f)
print(f"\nnumpy FFT와 일치: {np.allclose(F, F_numpy)}")

## 5. FFT와 Butterfly 연산

### 핵심 관찰: 짝수/홀수 분할

$f(x) = f_0 + f_1 x + f_2 x^2 + f_3 x^3$을 짝수/홀수 항으로 분리:

$$f(x) = \underbrace{(f_0 + f_2 x^2)}_{f_{\text{even}}(x^2)} + x \cdot \underbrace{(f_1 + f_3 x^2)}_{f_{\text{odd}}(x^2)}$$

즉: $f(x) = f_{\text{even}}(x^2) + x \cdot f_{\text{odd}}(x^2)$

### Butterfly 유도

$\omega^{n/2} = -1$을 이용하면, $k < n/2$에 대해:

$$f(\omega^k) = f_{\text{even}}(\omega^{2k}) + \omega^k \cdot f_{\text{odd}}(\omega^{2k})$$
$$f(\omega^{k+n/2}) = f_{\text{even}}(\omega^{2k}) - \omega^k \cdot f_{\text{odd}}(\omega^{2k})$$

(**$\omega^{k+n/2} = -\omega^k$**이므로)

이것이 **Butterfly 연산**입니다:

```
    f_even(ω^{2k})  ───────⊕──────→  f(ω^k)         = f_even + ω^k · f_odd
                        ╱ ╲
              × ω^k   ╱   ╲
                     ╱     ╲
    f_odd(ω^{2k})  ─•──────⊖──────→  f(ω^{k+n/2})   = f_even - ω^k · f_odd
```

**곱셈 1회** ($\omega^k \cdot f_{\text{odd}}$)와 **덧셈/뺄셈 각 1회**로 **2개의 출력**을 얻습니다.

In [ ]:
def fft_radix2(f):
    """Radix-2 Cooley-Tukey FFT (재귀 버전).
    
    각 단계에서 butterfly 연산을 수행합니다.
    """
    n = len(f)
    if n == 1:
        return f.copy()
    
    omega = np.exp(2j * np.pi / n)
    
    # 짝수/홀수 인덱스로 분할
    f_even = f[0::2]  # f_0, f_2, f_4, ...
    f_odd  = f[1::2]  # f_1, f_3, f_5, ...
    
    # 재귀적으로 FFT
    F_even = fft_radix2(f_even)
    F_odd  = fft_radix2(f_odd)
    
    # Butterfly 결합
    F = np.zeros(n, dtype=complex)
    for k in range(n // 2):
        twiddle = omega ** k * F_odd[k]  # twiddle factor 곱셈
        F[k]         = F_even[k] + twiddle  # 위쪽
        F[k + n // 2] = F_even[k] - twiddle  # 아래쪽
    
    return F

# 테스트
f = np.array([1, 2, 3, 4], dtype=complex)
F_our = fft_radix2(f)
F_ref = np.fft.fft(f)

print("Radix-2 FFT 결과 vs numpy FFT:")
print(f"{'k':>3} {'Our FFT':>20} {'numpy FFT':>20} {'일치':>5}")
print("-" * 52)
for k in range(len(f)):
    match = np.isclose(F_our[k], F_ref[k])
    print(f"{k:>3} {F_our[k]:>20.4f} {F_ref[k]:>20.4f} {'✓' if match else '✗':>5}")

### Butterfly 연산 단계별 시각화 (n=8)

In [ ]:
def fft_radix2_verbose(f):
    """FFT를 단계별로 보여주는 버전 (n=8)"""
    n = len(f)
    assert n == 8, "이 시각화는 n=8 전용"
    omega = np.exp(2j * np.pi / n)
    
    print(f"입력: f = {list(f.real.astype(int))}")
    print(f"ω = e^(2πi/8), ω = {omega:.4f}")
    print()
    
    # Stage 1: 길이 2의 butterfly (4쌍)
    # bit-reversal 순서: [0,4,2,6,1,5,3,7]
    br = [0, 4, 2, 6, 1, 5, 3, 7]
    x = f[br].copy()
    
    print("Bit-reversal 재배열:")
    print(f"  원래 인덱스: [0, 1, 2, 3, 4, 5, 6, 7]")
    print(f"  재배열:      {br}")
    print(f"  값:          {list(x.real.astype(int))}")
    print()
    
    # Stage 1: 길이 2 butterfly
    print("=" * 60)
    print("Stage 1: 길이 2 butterfly (twiddle = ω^0 = 1)")
    print("=" * 60)
    for i in range(0, 8, 2):
        a, b = x[i], x[i+1]
        x[i]   = a + b
        x[i+1] = a - b
        print(f"  ({a.real:.0f}, {b.real:.0f}) → ({x[i].real:.0f}, {x[i+1].real:.0f})")
    print(f"  결과: {[f'{v.real:.0f}' for v in x]}")
    print()
    
    # Stage 2: 길이 4 butterfly
    print("=" * 60)
    print("Stage 2: 길이 4 butterfly (twiddle = ω^0, ω^2)")
    print("=" * 60)
    for i in range(0, 8, 4):
        for k in range(2):
            tw = omega ** (2 * k)
            t = tw * x[i + k + 2]
            a = x[i + k]
            x[i + k]     = a + t
            x[i + k + 2] = a - t
            print(f"  k={k}: ω^{2*k} · {a.real:.0f} butterfly → ({x[i+k]:.2f}, {x[i+k+2]:.2f})")
    print(f"  결과: {[f'{v:.2f}' for v in x]}")
    print()
    
    # Stage 3: 길이 8 butterfly
    print("=" * 60)
    print("Stage 3: 길이 8 butterfly (twiddle = ω^0, ω^1, ω^2, ω^3)")
    print("=" * 60)
    for k in range(4):
        tw = omega ** k
        t = tw * x[k + 4]
        a = x[k]
        x[k]     = a + t
        x[k + 4] = a - t
        print(f"  k={k}: ω^{k} · x[{k+4}], butterfly → x[{k}]={x[k]:.2f}, x[{k+4}]={x[k+4]:.2f}")
    
    print(f"\n최종 결과: {[f'{v:.2f}' for v in x]}")
    return x

f8 = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=complex)
result = fft_radix2_verbose(f8)

print(f"\nnumpy FFT:   {[f'{v:.2f}' for v in np.fft.fft(f8)]}")
print(f"일치: {np.allclose(result, np.fft.fft(f8))}")

### Butterfly 다이어그램

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

n = 8
stages = int(np.log2(n))  # 3 stages

# Bit-reversal order
def bit_reverse(x, bits):
    result = 0
    for _ in range(bits):
        result = (result << 1) | (x & 1)
        x >>= 1
    return result

br_order = [bit_reverse(i, stages) for i in range(n)]

# 위치 설정
x_positions = np.linspace(0, 1, stages + 2)  # 입력 + stages + 출력
y_positions = np.linspace(0, 1, n)

# 입력 레이블
for i in range(n):
    ax.text(x_positions[0] - 0.05, y_positions[i], f'f[{br_order[i]}]',
            ha='right', va='center', fontsize=11, fontfamily='monospace')

# 출력 레이블
for i in range(n):
    ax.text(x_positions[-1] + 0.05, y_positions[i], f'F[{i}]',
            ha='left', va='center', fontsize=11, fontfamily='monospace')

# 수평선 (데이터 흐름)
for i in range(n):
    ax.plot([x_positions[0], x_positions[-1]], [y_positions[i], y_positions[i]],
            'gray', linewidth=0.5, alpha=0.3)

# Butterfly 연결선
colors = ['#e74c3c', '#2ecc71', '#3498db']
stage_labels = ['Stage 1\n(size 2)', 'Stage 2\n(size 4)', 'Stage 3\n(size 8)']

for stage in range(stages):
    block_size = 2 ** (stage + 1)
    half = block_size // 2
    x_mid = (x_positions[stage + 1] + x_positions[stage]) / 2 + 0.02
    
    ax.text(x_mid, 1.08, stage_labels[stage], ha='center', va='bottom',
            fontsize=10, color=colors[stage], fontweight='bold')
    
    for block_start in range(0, n, block_size):
        for k in range(half):
            top = block_start + k
            bot = block_start + k + half
            
            # butterfly 연결
            x1 = x_positions[stage] + 0.02
            x2 = x_positions[stage + 1] - 0.02
            
            # 직선 연결
            ax.annotate('', xy=(x2, y_positions[top]), xytext=(x1, y_positions[top]),
                       arrowprops=dict(arrowstyle='->', color=colors[stage], lw=1.5))
            ax.annotate('', xy=(x2, y_positions[bot]), xytext=(x1, y_positions[bot]),
                       arrowprops=dict(arrowstyle='->', color=colors[stage], lw=1.5))
            
            # 교차 연결 (butterfly)
            ax.annotate('', xy=(x2, y_positions[bot]), xytext=(x1, y_positions[top]),
                       arrowprops=dict(arrowstyle='->', color=colors[stage], lw=1, ls='--', alpha=0.6))
            ax.annotate('', xy=(x2, y_positions[top]), xytext=(x1, y_positions[bot]),
                       arrowprops=dict(arrowstyle='->', color=colors[stage], lw=1, ls='--', alpha=0.6))
            
            # twiddle factor 레이블
            tw_exp = k * (n // block_size)
            if tw_exp > 0:
                ax.text(x1 + 0.01, (y_positions[top] + y_positions[bot]) / 2,
                       f'ω^{tw_exp}', fontsize=8, color=colors[stage],
                       ha='left', va='center')

# 노드 점
for stage in range(stages + 1):
    for i in range(n):
        ax.plot(x_positions[stage], y_positions[i], 'ko', markersize=4)

ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.05, 1.15)
ax.axis('off')
ax.set_title('8-point FFT Butterfly Diagram (Cooley-Tukey, DIT)', fontsize=14, pad=30)

plt.tight_layout()
plt.savefig('/Users/jaeho/project/FAFFT/notebooks/butterfly_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

### 복잡도 분석

| | 나이브 DFT | FFT (Cooley-Tukey) |
|---|---|---|
| 곱셈 수 | $n^2$ | $\frac{n}{2} \log_2 n$ |
| 덧셈 수 | $n(n-1)$ | $n \log_2 n$ |
| **총 복잡도** | **$O(n^2)$** | **$O(n \log n)$** |

예: $n = 2^{16} = 65536$
- 나이브: $\approx 4.3 \times 10^9$ 연산
- FFT: $\approx 1 \times 10^6$ 연산 → **약 4000배 빠름!**

In [ ]:
# 복잡도 비교 시각화
ns = np.array([2**k for k in range(4, 18)])
naive_ops = ns ** 2
fft_ops = ns * np.log2(ns)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(np.log2(ns), naive_ops, 'r-o', label='Naive DFT: $O(n^2)$', markersize=5)
ax.semilogy(np.log2(ns), fft_ops, 'b-o', label='FFT: $O(n \\log n)$', markersize=5)

# HQC 파라미터 표시
hqc_n = {1: 17669, 3: 35851, 5: 57637}
for level, n_val in hqc_n.items():
    log_n = np.log2(n_val)
    ax.axvline(x=log_n, color='green', linestyle=':', alpha=0.5)
    ax.text(log_n, 1e3, f'HQC-{level}\n(n={n_val})', ha='center', fontsize=8, color='green')

ax.set_xlabel('$\\log_2(n)$')
ax.set_ylabel('Number of operations')
ax.set_title('Naive DFT vs FFT Complexity')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/jaeho/project/FAFFT/notebooks/complexity_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("HQC-1 (n=17669) 기준:")
n_hqc = 17669
print(f"  나이브: {n_hqc**2:>15,} 연산")
print(f"  FFT:    {int(n_hqc * np.log2(n_hqc)):>15,} 연산")
print(f"  비율:   {n_hqc**2 / (n_hqc * np.log2(n_hqc)):>15,.0f}배 차이")

## 6. FFT를 이용한 다항식 곱셈

전체 과정을 조합하면:

In [ ]:
def fft_multiply(A, B):
    """FFT를 이용한 다항식 곱셈.
    
    1. 길이를 2의 거듭제곱으로 패딩
    2. FFT로 evaluate
    3. Pointwise multiply
    4. Inverse FFT로 interpolate
    """
    # 결과 다항식의 길이
    result_len = len(A) + len(B) - 1
    
    # 2의 거듭제곱으로 올림
    N = 1
    while N < result_len:
        N *= 2
    
    # Zero-padding
    A_padded = np.zeros(N, dtype=complex)
    B_padded = np.zeros(N, dtype=complex)
    A_padded[:len(A)] = A
    B_padded[:len(B)] = B
    
    print(f"Step 0: Zero-padding to N={N}")
    print(f"  A = {list(A)} → {list(A_padded.real.astype(int))}")
    print(f"  B = {list(B)} → {list(B_padded.real.astype(int))}")
    print()
    
    # Step 1: FFT (Evaluate)
    A_fft = np.fft.fft(A_padded)
    B_fft = np.fft.fft(B_padded)
    print("Step 1: FFT (Evaluate at roots of unity)")
    print(f"  A_hat = [{', '.join(f'{v:.1f}' for v in A_fft)}]")
    print(f"  B_hat = [{', '.join(f'{v:.1f}' for v in B_fft)}]")
    print()
    
    # Step 2: Pointwise Multiply
    C_fft = A_fft * B_fft
    print("Step 2: Pointwise Multiply")
    print(f"  C_hat = [{', '.join(f'{v:.1f}' for v in C_fft)}]")
    print()
    
    # Step 3: Inverse FFT (Interpolate)
    C = np.fft.ifft(C_fft).real
    C = np.round(C[:result_len]).astype(int)
    print("Step 3: Inverse FFT (Interpolate)")
    print(f"  C = {list(C)}")
    
    return list(C)

print("FFT 기반 다항식 곱셈")
print("=" * 60)
A = [2, 3, 1]   # 2 + 3x + x^2
B = [1, 2]       # 1 + 2x
print(f"A(x) = {poly_to_str(A)}")
print(f"B(x) = {poly_to_str(B)}")
print()

C_fft = fft_multiply(A, B)
C_naive = [2, 7, 7, 2]  # 앞에서 계산한 결과

print(f"\n나이브 결과: {C_naive}")
print(f"FFT 결과:    {C_fft}")
print(f"일치: {'✓' if C_fft == C_naive else '✗'}")

## 7. 왜 $\mathbb{F}_2$에서는 이 방법이 안 되는가?

### 문제 1: Root of Unity가 없다

$\mathbb{F}_2 = \{0, 1\}$에서:
- $1^n = 1$이므로, 모든 $n$에 대해 $\omega = 1$뿐
- 하지만 $\omega^k = 1$ for all $k$이므로, 서로 다른 점을 만들 수 없음

### 문제 2: 확장체에서도 안 된다

$\mathbb{F}_{2^m}$의 multiplicative group $\mathbb{F}_{2^m}^*$의 크기는 $2^m - 1$ (항상 **홀수**).

FFT에는 $2^k$-th root of unity가 필요한데, $2^m - 1$이 홀수이므로
$2^k$로 나눠지지 않습니다. 따라서 $2^k$-th root of unity가 존재하지 않습니다.

### 문제 3: $\omega^{n/2} = -1$이 성립하지 않는다

$\mathbb{F}_2$에서 $-1 = 1$이므로, butterfly의 핵심인 "$+$와 $-$의 대칭"이 사라집니다.

$$f(\omega^k) = f_{\text{even}} + \omega^k \cdot f_{\text{odd}}$$
$$f(\omega^{k+n/2}) = f_{\text{even}} - \omega^k \cdot f_{\text{odd}} = f_{\text{even}} + \omega^k \cdot f_{\text{odd}}$$

두 식이 **같아져 버립니다!** Butterfly가 무의미해집니다.

In [ ]:
print("F_2에서 일반 FFT가 실패하는 이유")
print("=" * 60)

print("\n1. Root of Unity 문제:")
print("   F_2 = {0, 1}")
print("   1^n = 1 for all n → ω = 1이 유일한 후보")
print("   evaluation points = {1, 1, 1, ...} → 서로 다른 점이 아님!")

print("\n2. 확장체 F_{2^m}에서도 실패:")
for m in [4, 8, 16, 32]:
    group_size = 2**m - 1
    # 2^k가 group_size를 나누는지 확인
    max_pow2 = 0
    temp = group_size
    while temp % 2 == 0:
        max_pow2 += 1
        temp //= 2
    print(f"   F_{{2^{m:2d}}}: |F*| = 2^{m}-1 = {group_size:>12,} (홀수 → 2^k-th root of unity 없음)")

print("\n3. Butterfly 붕괴:")
print("   일반 FFT butterfly:")
print("     f(ω^k)       = f_even + ω^k · f_odd")
print("     f(ω^{k+n/2}) = f_even - ω^k · f_odd")
print("")
print("   F_2에서 -1 = 1이므로:")
print("     f(ω^k)       = f_even + ω^k · f_odd")
print("     f(ω^{k+n/2}) = f_even + ω^k · f_odd  ← 같은 값!")
print("")
print("   → 2개의 다른 출력을 만들 수 없음 → FFT 분할 정복 불가능")

## 8. 해결의 방향: Additive FFT

일반 FFT는 **multiplicative** 구조 (root of unity, 곱셈군)에 의존합니다.

$\mathbb{F}_2$에서는 multiplicative 구조가 FFT에 적합하지 않지만,
**additive** 구조 (덧셈 부분군, XOR)는 풍부합니다!

| | 일반 FFT | Additive FFT |
|---|---|---|
| **평가 점** | $\{1, \omega, \omega^2, \ldots\}$ (multiplicative coset) | $a + V_i$ (additive coset) |
| **분할 기준** | $\omega^{n/2} = -1$ | $V_i = V_{i-1} \cup (v_{i-1} + V_{i-1})$ |
| **Butterfly** | $f_{\text{even}} \pm \omega^k \cdot f_{\text{odd}}$ | $f_l + s_{i-1}(a) \cdot f_h$ |
| **필요 조건** | primitive root of unity | **Cantor Basis** |
| **적용 체** | $\mathbb{C}$, $\mathbb{F}_p$ ($p$ 홀수) | $\mathbb{F}_{2^m}$ |

### 핵심 통찰

> 일반 FFT의 butterfly가 $\omega^{n/2} = -1$에서 $+/-$ 대칭을 이용한다면,
> Additive FFT의 butterfly는 **vanishing polynomial $s_i$의 선형성**과
> **$s_i(v) = 0$ for $v \in V_i$**라는 성질을 이용합니다.

다음 노트북 (`01_cantor_basis.ipynb`)에서 Cantor Basis와 Additive FFT를 자세히 다룹니다.

In [ ]:
# 요약 다이어그램
print("학습 로드맵")
print("=" * 60)
print()
print("  [00_fft_basics] ← 지금 여기")
print("    │")
print("    │  다항식 곱셈 → FFT → Butterfly")
print("    │  왜 F_2에서 안 되는가?")
print("    │")
print("    ▼")
print("  [01_cantor_basis]")
print("    │")
print("    │  Cantor Basis → 부분공간 V_i")
print("    │  Vanishing polynomial s_i")
print("    │  Additive FFT butterfly")
print("    │")
print("    ▼")
print("  [02_fafft] (다음)")
print("    │")
print("    │  Frobenius map (x → x²)")
print("    │  FAFFT: evaluation point 축소")
print("    │  다항식 곱셈 적용")
print("    │")
print("    ▼")
print("  [논문 읽기]")
print("    │")
print("    ├── IEEE IoT25: Cortex-M4 최적화")
print("    └── TCHES26: CRT 결합 + 다중 플랫폼")